# Co-teaching under noisy/imbalanced labels — GPU verification on Colab

This notebook clones **[oai-noisy-labels-coteaching](https://github.com/michal-matusik/oai-noisy-labels-coteaching)** and runs the exact same training code (`src.train.run`, unmodified) on a Colab GPU.

**Why this exists:** a local CPU re-run of this exact code got mean BAC 0.7725 (90.8/100), below the reference notebook's GPU run (0.8812, 100/100). Both runs are deterministic on their own hardware, but CPU vs. GPU floating-point kernels diverge over hundreds of training steps even from the same seed — so the two numbers aren't directly comparable. This notebook re-runs the identical code on GPU to check whether it lands close to the reference's 100/100 there, which would confirm the gap is a CPU-only verification artifact rather than anything wrong with the ported solution. See `SOLUTION.md` §3 for the full diagnosis.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — set Runtime > Change runtime type > GPU")

In [ ]:
!git clone https://github.com/michal-matusik/oai-noisy-labels-coteaching.git
%cd oai-noisy-labels-coteaching

In [ ]:
# Colab already ships a CUDA-enabled torch/torchvision — only install the extra deps
!pip install -q gdown pandas scikit-learn

## Download the data and run training

Same data source ids as `configs/config.yaml`, same `src.train.run` call the local CPU verification used (`num_epochs=6`, `lr=1e-2`, `batch_size=128`, `seed=123`) — nothing about the training code changes between CPU and GPU, only the `device`, which `run()` auto-selects (`"cuda" if torch.cuda.is_available() else "cpu"`).

In [ ]:
from src.dataset import download_data, unpack_data

download_data('train', '1qmNNmDv-wUcAv5mvO6vYJV3mQ2SNIGnI')
download_data('val', '1YUJYD12NmKRSzFJGMrX-a61d6mnTaWbG')
unpack_data('.', 'train')
unpack_data('.', 'val')

In [ ]:
import json

from src.dataset import load_data
from src.evaluate import performance, predict_and_evaluate
from src.train import run

model1, model2 = run(train_path="train", val_path="val")

_, val_loader = load_data("train", "val", 128)
bac1 = predict_and_evaluate(model1, val_loader, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
bac2 = predict_and_evaluate(model2, val_loader, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
mean_bac = (bac1 + bac2) / 2
score = 0.0 if mean_bac <= 0.5 else (100.0 if mean_bac >= 0.8 else 100 * (mean_bac - 0.5) / 0.3)

result = {"bac_model1": bac1, "bac_model2": bac2, "mean_bac": mean_bac, "score": score}
print(json.dumps(result, indent=2))

with open("results/gpu_run_results.json", "w") as f:
    json.dump(result, f, indent=2)

## Download the result to keep it

If this lands near 100/100, it confirms the local CPU number (90.8/100) was purely a hardware/kernel-numerics artifact, not a bug — worth committing `results/gpu_run_results.json` back into the repo as the authoritative GPU-verified number.

In [ ]:
from google.colab import files
files.download("results/gpu_run_results.json")